# 05. Dataset и DataLoader — ч.2 (практика)

Видео: [5. Подготовка данных ч.2. Dataset и DataLoader](https://www.youtube.com/watch?v=ztTlRgwdhkU&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=6)

Задачи **не повторяют** код автора один в один: вместо MNIST — свой маленький набор «фрукты», который создаёт ячейка ниже. Структура похожа (папка = класс), но есть отличия, под которые код придётся думать самому: мусорные файлы, тензор вместо numpy, дисбаланс, Windows-пути.

**Как решать.** Замени `...` на код → запусти проверку под ним. Решения — в `05_dataset_dataloader_2_решения.ipynb`, только после своей попытки.

| блок | о чём | где в видео |
|:---|:---|:---|
| 1 | разминка: `Dataset` на тензорах, `DataLoader` | 2:36–3:08, 14:38 |
| 2 | обходим папки `os.walk` | 3:41–5:45 |
| 3 | свой `FruitsDataset` | 5:45–9:56 |
| 4 | индекс → имя класса (домашка автора) | 13:03–13:35 |
| 5 | `transform` в конструкторе | 8:52 |
| 6 | `random_split` и три `DataLoader` | 14:07–15:41 |
| 7 | ловушки | — |
| 8 | то же на кликах для рекомендаций | — |

In [ ]:
import os
import json
import shutil

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split, RandomSampler, SequentialSampler

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"

## 0. Данные

Ячейка ниже создаёт рядом с ноутбуком папку `_data/`:

```text
_data/
├── fruits/
│   ├── train/
│   │   ├── README.txt          ← мусор, не класс
│   │   ├── apple/  12 картинок
│   │   ├── pear/   10 картинок + notes.txt (мусор)
│   │   └── plum/    3 картинки  ← мало!
│   └── test/
│       ├── apple/ pear/ plum/  по 4 картинки
└── clicks.json                 ← для блока 8
```

Картинки 8×8, серые. У каждого класса своя яркость (apple тёмный, pear средний, plum светлый) — так проверки узнают, что метка совпадает с картинкой. Запусти ячейку и загляни в папку глазами.

In [ ]:
DATA = "_data"
FRUITS = os.path.join(DATA, "fruits")
TRAIN_DIR = os.path.join(FRUITS, "train")
TEST_DIR = os.path.join(FRUITS, "test")
CLICKS_PATH = os.path.join(DATA, "clicks.json")

SIZE = 8
BRIGHTNESS = {"apple": 40, "pear": 120, "plum": 200}
COUNTS = {
    "train": {"apple": 12, "pear": 10, "plum": 3},
    "test": {"apple": 4, "pear": 4, "plum": 4},
}


def make_data():
    shutil.rmtree(DATA, ignore_errors=True)
    rng = np.random.default_rng(0)
    for split, per_class in COUNTS.items():
        for name, n in per_class.items():
            folder = os.path.join(FRUITS, split, name)
            os.makedirs(folder)
            for k in range(n):
                noise = rng.integers(-20, 21, size=(SIZE, SIZE))
                img = np.clip(BRIGHTNESS[name] + noise, 0, 255).astype(np.uint8)
                Image.fromarray(img).save(os.path.join(folder, f"{name}_{k:03d}.png"))
    with open(os.path.join(TRAIN_DIR, "README.txt"), "w", encoding="utf-8") as f:
        f.write("описание датасета")
    with open(os.path.join(TRAIN_DIR, "pear", "notes.txt"), "w", encoding="utf-8") as f:
        f.write("не картинка")

    clicks = [
        {"user": "u_17", "item": "sku_501", "click": 1},
        {"user": "u_17", "item": "sku_777", "click": 0},
        {"user": "u_03", "item": "sku_501", "click": 1},
        {"user": "u_42", "item": "sku_123", "click": 1},
        {"user": "u_42", "item": "sku_501", "click": 0},
        {"user": "u_03", "item": "sku_777", "click": 1},
        {"user": "u_99", "item": "sku_123", "click": 0},
        {"user": "u_17", "item": "sku_123", "click": 1},
    ]
    with open(CLICKS_PATH, "w", encoding="utf-8") as f:
        json.dump(clicks, f, ensure_ascii=False, indent=2)


make_data()
print(sorted(os.listdir(TRAIN_DIR)))
print(sorted(os.listdir(os.path.join(TRAIN_DIR, "pear")))[:3], "...")

## 1. Разминка: свой Dataset на тензорах
*видео 2:36–3:08*

**Теория.** Свой датасет — класс-наследник `torch.utils.data.Dataset` с тремя методами:

| метод | что делает |
|:---|:---|
| `__init__(...)` | запоминает, **откуда** брать примеры (тензоры, пути к файлам, transform) |
| `__len__(self)` | сколько примеров — по нему `DataLoader` понимает, сколько батчей |
| `__getitem__(self, idx)` | возвращает **один** пример: пару `(данные, ответ)` |

`ds[i]` вызывает `ds.__getitem__(i)`, `len(ds)` — `ds.__len__()`.

**1.1** Допиши `ToyDataset`: хранит `X` формы `(N, F)` и `y` формы `(N,)`; `ds[i]` → `(X[i], y[i])`.

**Ожидаемый вывод:**

```text
20 torch.Size([8]) tensor(...)    ← в конце метка класса, число от 0 до 4
```

In [ ]:
X_all = torch.randn(20, 8)           # 20 примеров по 8 признаков
y_all = torch.randint(0, 5, (20,))   # 5 классов

In [ ]:
class ToyDataset(Dataset):
    def __init__(self, X, y):
        ...

    def __len__(self):
        ...

    def __getitem__(self, idx):
        ...


toy_ds = ToyDataset(X_all, y_all)
x0, y0 = toy_ds[0]
print(len(toy_ds), x0.shape, y0)

In [ ]:
need(isinstance(toy_ds, Dataset), "класс должен наследоваться от Dataset")
need(len(toy_ds) == 20, f"len={len(toy_ds)}, ждали 20")
need_tensor(x0, "x0")
need(x0.shape == (8,), f"x0.shape={tuple(x0.shape)}")
need(torch.equal(x0, X_all[0]) and int(y0) == int(y_all[0]), "toy_ds[0] должен совпадать с X_all[0], y_all[0]")
x19, y19 = toy_ds[19]
need(torch.equal(x19, X_all[19]) and int(y19) == int(y_all[19]), "проверь крайний индекс 19")
print("1.1 — верно")

**Теория.** `DataLoader` берёт датасет и выдаёт **батчи**: вызывает `ds[i]` для `batch_size` индексов и склеивает результаты по новой оси 0 (как `torch.stack`).

```python
loader = DataLoader(ds, batch_size=4, shuffle=True)
for xb, yb in loader:   # xb: (4, F), yb: (4,)
    ...
```

`shuffle=True` — на каждой эпохе новый порядок (для train). `next(iter(loader))` — взять один батч без цикла.

**1.2** Собери `toy_loader` (`batch_size=4`, `shuffle=True`) и возьми из него один батч `xb`, `yb`.

**Ожидаемый вывод:**

```text
torch.Size([4, 8]) torch.Size([4])
```

In [ ]:
toy_loader = ...
xb, yb = ...
print(xb.shape, yb.shape)

In [ ]:
need(isinstance(toy_loader, DataLoader), "нужен DataLoader")
need(toy_loader.batch_size == 4, f"batch_size={toy_loader.batch_size}")
need(isinstance(toy_loader.sampler, RandomSampler), "для train нужен shuffle=True")
need_tensor(xb, "xb")
need(xb.shape == (4, 8), f"xb.shape={tuple(xb.shape)}, ждали (4, 8)")
need(yb.shape == (4,), f"yb.shape={tuple(yb.shape)}, ждали (4,)")
print("1.2 — верно")

## 2. Обходим папки: `os.walk`
*видео 3:41–5:45*

**Теория.** `os.walk(root)` обходит папку и все вложенные. На каждом шаге отдаёт тройку:

```python
for dirpath, dirnames, filenames in os.walk(root):
    # dirpath   — путь к текущей папке
    # dirnames  — имена вложенных папок
    # filenames — имена файлов прямо в этой папке
```

Первый шаг — сама `root` (у нас там папки-классы и `README.txt`), дальше — каждая папка класса.

Отличие от автора: он берёт имя класса через `dirpath.split("/")[-1]`. На Windows разделитель `\`, и это сломается. Надёжно: `os.path.basename(dirpath)`.

**2.1** Пройди `TRAIN_DIR` через `os.walk` и собери словарь `counts`: имя класса → число **`.png`** файлов. Саму `TRAIN_DIR` пропусти, мусорные `.txt` не считай.

**Ожидаемый вывод:**

```text
{'apple': 12, 'pear': 10, 'plum': 3}
```

In [ ]:
counts = {}
...
print(counts)

In [ ]:
need(counts == {"apple": 12, "pear": 10, "plum": 3}, f"получили {counts}; .txt не считаются, README не класс")
print("2.1 — верно")

**2.2** Вспомни видео 4 про баланс. Посчитай `imbalance_ratio` = max / min по `counts` и реши, сбалансирован ли train: `is_balanced = ...` (`True` / `False`).

**Ожидаемый вывод:**

```text
4.0 False
```

In [ ]:
imbalance_ratio = ...
is_balanced = ...
print(imbalance_ratio, is_balanced)

In [ ]:
need(abs(imbalance_ratio - 4.0) < 1e-6, f"ratio={imbalance_ratio}, ждали 12 / 3 = 4.0")
need(is_balanced is False, "разница в 4 раза — это уже не «плюс-минус одинаково»")
print("2.2 — верно")

## 3. Свой `FruitsDataset`
*видео 5:45–9:56*

**Теория.** Тот же каркас, что в разминке, но данные — файлы на диске:

- в `__init__` собираем **только пути** и метки: список `data_list` из пар `(путь_к_png, индекс_класса)`. Сами картинки не читаем;
- в `__getitem__(idx)` берём `data_list[idx]`, **читаем файл** и возвращаем `(картинка, метка)`.

Так датасет может быть хоть 100 ГБ: в память попадает только то, что запросил `DataLoader`.

Нужные функции:

```python
os.listdir(folder)                 # имена всего внутри папки
os.path.isdir(path)                # это папка?
os.path.join(folder, name)         # склеить путь
with Image.open(path) as im:       # PIL; with — чтобы файл сразу закрылся
    arr = np.array(im)             # (H, W), uint8, значения 0..255
torch.tensor(arr, dtype=torch.float32)
```

**3.1** Напиши `FruitsDataset(root, transform=None)`. Требования (часть — **не как у автора**):

| что | требование |
|:---|:---|
| `self.classes` | **отсортированный** список имён папок-классов (`README.txt` — не класс) |
| `self.class_to_idx` | имя → индекс по порядку `classes` |
| `self.data_list` | пары `(путь, индекс_класса)`, только `.png` |
| `__getitem__` | возвращает `(x, label)`: `x` — **тензор** `float32` формы `(1, H, W)` со значениями 0..1, `label` — `int` |
| `transform` | если задан — применить к `x` перед возвратом |

Подсказка к форме: `(H, W)` → `(1, H, W)` — это `unsqueeze(0)` (ось каналов, как в видео 4).

**Ожидаемый вывод:**

```text
['apple', 'pear', 'plum'] {'apple': 0, 'pear': 1, 'plum': 2}
25 12 torch.Size([1, 8, 8]) torch.float32 0
```

In [ ]:
class FruitsDataset(Dataset):
    def __init__(self, root, transform=None):
        ...

    def __len__(self):
        ...

    def __getitem__(self, idx):
        ...


train_ds = FruitsDataset(TRAIN_DIR)
test_ds = FruitsDataset(TEST_DIR)
x, label = train_ds[0]
print(train_ds.classes, train_ds.class_to_idx)
print(len(train_ds), len(test_ds), x.shape, x.dtype, label)

In [ ]:
need(isinstance(train_ds, Dataset), "наследуйся от Dataset")
need(train_ds.classes == ["apple", "pear", "plum"], f"classes={train_ds.classes}: отсортируй и выкинь не-папки")
need(train_ds.class_to_idx == {"apple": 0, "pear": 1, "plum": 2}, f"class_to_idx={train_ds.class_to_idx}")
need(len(train_ds) == 25, f"len(train_ds)={len(train_ds)}, ждали 12+10+3=25 (notes.txt не картинка)")
need(len(test_ds) == 12, f"len(test_ds)={len(test_ds)}, ждали 12")
need(test_ds.class_to_idx == train_ds.class_to_idx, "у train и test должно быть одинаковое соответствие класс → индекс")
need(all(isinstance(p, str) and p.endswith(".png") for p, _ in train_ds.data_list),
     "в data_list должны быть пути к .png, а не сами картинки")

need_tensor(x, "x")
need(x.shape == (1, 8, 8), f"x.shape={tuple(x.shape)}, ждали (1, 8, 8)")
need(x.dtype == torch.float32, f"x.dtype={x.dtype}")
need(0.0 <= x.min().item() and x.max().item() <= 1.0, "значения должны быть в 0..1 (раздели на 255)")
need(isinstance(label, int), f"label должен быть int, а не {type(label).__name__}")

_levels = torch.tensor([40, 120, 200]) / 255
for _i in range(len(train_ds)):
    _x, _y = train_ds[_i]
    _guess = (_levels - _x.mean()).abs().argmin().item()
    need(_guess == _y, f"train_ds[{_i}]: метка {_y}, а по яркости картинка похожа на класс {_guess}")
print("3.1 — верно")

## 4. Индекс → имя класса
*видео 13:03–13:35 — автор оставил это как домашку*

**Теория.** `class_to_idx` отвечает на вопрос «какой индекс у pear?». Чтобы по предсказанию сети (индекс) показать человеку имя, нужен **обратный** словарь.

**4.1** Построй `idx_to_class` из `train_ds.class_to_idx` и напиши `name_of(ds, i)` — имя класса `i`-го примера.

**Ожидаемый вывод:**

```text
{0: 'apple', 1: 'pear', 2: 'plum'}
apple plum
```

In [ ]:
idx_to_class = ...


def name_of(ds, i):
    ...


print(idx_to_class)
print(name_of(train_ds, 0), name_of(train_ds, 24))

In [ ]:
need(idx_to_class == {0: "apple", 1: "pear", 2: "plum"}, f"idx_to_class={idx_to_class}")
for _i in range(len(train_ds)):
    _folder = os.path.basename(os.path.dirname(train_ds.data_list[_i][0]))
    need(name_of(train_ds, _i) == _folder, f"name_of(train_ds, {_i}) не совпало с папкой {_folder}")
print("4.1 — верно")

## 5. `transform` в конструкторе
*видео 8:52*

**Теория.** `transform` — любая функция «картинка → картинка». Датасет её не знает заранее, просто вызывает, если передали. Так один класс работает с разной предобработкой. В видео 7 будут готовые трансформации из `torchvision`, а пока — своя функция.

**5.1** Создай `train_ds_norm` — тот же train, но с `transform`, который переводит значения из `0..1` в `-1..1` (`x * 2 - 1`).

**Ожидаемый вывод:**

```text
-0.84... -0.52...    ← два числа между -1 и 1
```

In [ ]:
train_ds_norm = ...
xn, _ = train_ds_norm[0]
print(xn.min().item(), xn.max().item())

In [ ]:
need(isinstance(train_ds_norm, FruitsDataset), "это должен быть FruitsDataset с transform")
need(torch.allclose(train_ds_norm[0][0], train_ds[0][0] * 2 - 1), "train_ds_norm[0] должен быть train_ds[0] * 2 - 1")
need(train_ds_norm[0][1] == train_ds[0][1], "transform не должен трогать метку")
print("5.1 — верно")

## 6. `random_split` и три `DataLoader`
*видео 14:07–15:41*

**Теория.** Отдельного val у нас нет — отрезаем его от train:

```python
part_a, part_b = random_split(ds, [0.8, 0.2], generator=torch.Generator().manual_seed(42))
```

- доли (или точные длины `[20, 5]`) должны давать в сумме весь датасет;
- `generator` с seed — чтобы разбиение было одинаковым при каждом запуске;
- результат — `Subset`: не копия данных, а «датасет + список индексов» (`part_a.indices`).

**6.1** Раздели `train_ds` на `train_part` / `val_part` в пропорции 80 / 20 с seed 42.

**Ожидаемый вывод:**

```text
20 5
```

In [ ]:
train_part, val_part = ...
print(len(train_part), len(val_part))

In [ ]:
need(len(train_part) == 20 and len(val_part) == 5, f"длины {len(train_part)}, {len(val_part)}; ждали 20 и 5")
_a, _b = set(train_part.indices), set(val_part.indices)
need(not (_a & _b), "train и val пересекаются")
need(_a | _b == set(range(25)), "вместе train и val должны покрыть все 25 примеров")
_again, _ = random_split(train_ds, [0.8, 0.2], generator=torch.Generator().manual_seed(42))
need(list(_again.indices) == list(train_part.indices), "передай generator с seed 42 — иначе разбиение каждый раз новое")
print("6.1 — верно")

**6.2** Сначала **в уме**, потом кодом.

При `batch_size=6` сколько будет батчей и какого размера последний для `train_part` (20), `val_part` (5), `test_ds` (12)? Заполни `pred` парами `(число_батчей, размер_последнего)`.

Затем создай `train_loader`, `val_loader`, `test_loader` с `batch_size=6`: train перемешивается, val и test — нет.

**Ожидаемый вывод:**

```text
torch.Size([6, 1, 8, 8]) torch.Size([6]) torch.int64
```

In [ ]:
pred = {
    "train": ...,
    "val": ...,
    "test": ...,
}

train_loader = ...
val_loader = ...
test_loader = ...

xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape, yb.dtype)

In [ ]:
_loaders = {"train": train_loader, "val": val_loader, "test": test_loader}
for _name, _loader in _loaders.items():
    _sizes = [bx.shape[0] for bx, _ in _loader]
    need(pred[_name] == (len(_sizes), _sizes[-1]), f"{_name}: ты предсказал {pred[_name]}, на деле {(len(_sizes), _sizes[-1])}")
need(isinstance(train_loader.sampler, RandomSampler), "train_loader: нужен shuffle=True")
need(isinstance(val_loader.sampler, SequentialSampler), "val_loader: перемешивать не нужно")
need(isinstance(test_loader.sampler, SequentialSampler), "test_loader: перемешивать не нужно")
need(xb.shape == (6, 1, 8, 8), f"xb.shape={tuple(xb.shape)}, ждали (6, 1, 8, 8) = (B, C, H, W)")
need(yb.shape == (6,) and yb.dtype == torch.int64, f"yb: {tuple(yb.shape)}, {yb.dtype}; int-метки DataLoader склеивает в int64")
print("6.2 — верно")

## 7. Ловушки

**7.1** Что будет при `train_ds[len(train_ds)]`? Запиши **класс исключения** (не строку) в `err_type`. Сначала подумай, на какой строке `__getitem__` упадёт, потом проверь.

**Ожидаемый вывод:**

```text
(ячейка ничего не печатает — смотри проверку ниже)
```

In [ ]:
err_type = ...

In [ ]:
need(isinstance(err_type, type) and issubclass(err_type, BaseException), "нужен класс исключения, например ValueError")
try:
    train_ds[len(train_ds)]
    need(False, "не упало — странно, проверь __getitem__")
except AssertionError:
    raise
except Exception as e:
    need(type(e) is err_type, f"на деле {type(e).__name__}, а ты написал {err_type.__name__}")
print("7.1 — верно")

**7.2** Автор достаёт имя класса из пути через `split("/")`. Напиши `class_from_path(path)` так, чтобы работало и с путями из `os.path.join` (на Windows там `\`), и с путями через `/`.

**Ожидаемый вывод:**

```text
apple plum
```

In [ ]:
def class_from_path(path):
    ...


print(class_from_path(train_ds.data_list[0][0]), class_from_path("data/test/plum/x.png"))

In [ ]:
need(class_from_path(os.path.join(TRAIN_DIR, "pear", "pear_000.png")) == "pear", "путь из os.path.join")
need(class_from_path("data/test/plum/x.png") == "plum", "путь через /")
print("7.2 — верно")

**7.3** Пройди `train_loader` два раза подряд (две «эпохи») и `val_loader` два раза. Предскажи: совпадёт ли порядок примеров между эпохами?

`same_order_train`, `same_order_val` — `True` / `False`. Проверка пройдёт обе эпохи сама.

**Ожидаемый вывод:**

```text
(ячейка ничего не печатает — смотри проверку ниже)
```

In [ ]:
same_order_train = ...
same_order_val = ...

In [ ]:
def _epoch(loader):
    return torch.cat([bx for bx, _ in loader])


torch.manual_seed(1)
_real_train = torch.equal(_epoch(train_loader), _epoch(train_loader))
_real_val = torch.equal(_epoch(val_loader), _epoch(val_loader))
need(same_order_train == _real_train, f"train: на деле порядок {'совпал' if _real_train else 'разный'}")
need(same_order_val == _real_val, f"val: на деле порядок {'совпал' if _real_val else 'разный'}")
print("7.3 — верно")

## 8. То же на кликах (рекомендации)
*не из видео — перенос на другой тип данных*

**Теория.** В `_data/clicks.json` — список событий `{"user": "u_17", "item": "sku_501", "click": 1}`. Каркас `Dataset` тот же, отличия:

- файл маленький — его можно прочитать целиком в `__init__` (`json.load`). Ленивое чтение нужно, когда каждый пример — тяжёлый файл;
- id — строки (`"u_17"`), а `nn.Embedding` (будет позже) понимает только индексы `0..n-1`. Поэтому, как `class_to_idx`, строим `user2idx` и `item2idx` по **отсортированным** уникальным id.

**8.1** Напиши `ClicksDataset(path)`:

| что | требование |
|:---|:---|
| `self.user2idx`, `self.item2idx` | отсортированные уникальные id → `0..n-1` |
| `__getitem__` | `((user_idx, item_idx), click)`: два тензора `int64` формы `()` и тензор `float32` формы `()` |

**Ожидаемый вывод:**

```text
{'u_03': 0, 'u_17': 1, 'u_42': 2, 'u_99': 3} {'sku_123': 0, 'sku_501': 1, 'sku_777': 2}
tensor([1, 1, 0, 2]) tensor([1, 2, 1, 0]) tensor([1., 0., 1., 1.])
```

In [ ]:
class ClicksDataset(Dataset):
    def __init__(self, path):
        ...

    def __len__(self):
        ...

    def __getitem__(self, idx):
        ...


clicks_ds = ClicksDataset(CLICKS_PATH)
clicks_loader = DataLoader(clicks_ds, batch_size=4, shuffle=False)
(bu, bi), by = next(iter(clicks_loader))
print(clicks_ds.user2idx, clicks_ds.item2idx)
print(bu, bi, by)

In [ ]:
need(len(clicks_ds) == 8, f"len={len(clicks_ds)}, ждали 8")
need(clicks_ds.user2idx == {"u_03": 0, "u_17": 1, "u_42": 2, "u_99": 3}, f"user2idx={clicks_ds.user2idx}")
need(clicks_ds.item2idx == {"sku_123": 0, "sku_501": 1, "sku_777": 2}, f"item2idx={clicks_ds.item2idx}")
(_u, _i), _c = clicks_ds[0]
need_tensor(_u, "user")
need(_u.shape == () and _u.dtype == torch.int64, f"user: форма {tuple(_u.shape)}, {_u.dtype}; ждали (), int64")
need(_i.dtype == torch.int64 and _c.dtype == torch.float32, f"item {_i.dtype}, click {_c.dtype}")
need((_u.item(), _i.item(), _c.item()) == (1, 1, 1.0), "clicks_ds[0]: u_17 → 1, sku_501 → 1, click 1.0")
need(bu.tolist() == [1, 1, 0, 2], f"bu={bu.tolist()}")
need(bi.tolist() == [1, 2, 1, 0], f"bi={bi.tolist()}")
need(by.tolist() == [1.0, 0.0, 1.0, 1.0], f"by={by.tolist()}")
print("8.1 — верно")

## 9. Что унести из видео 5

1. Свой `Dataset`: в `__init__` — пути и метки, в `__getitem__` — чтение одного файла. Картинки целиком в память не грузим.
2. `classes` сортируем — тогда train и test получают одно и то же `class_to_idx`.
3. Мусорные файлы фильтруем (`endswith(".png")`, `os.path.isdir`).
4. Пути — через `os.path.join` / `os.path.basename`, не через `split("/")`.
5. `random_split` с `generator` — воспроизводимое train / val.
6. `DataLoader`: train `shuffle=True`, val / test — `False`; последний батч может быть короче.
7. Тот же каркас работает для любых данных: картинки, клики, таблицы.

**Сразу после этого ноутбука** — свой проект: `project/fashion_mnist.ipynb`, раздел «Видео 5». То же самое, что у автора на MNIST, но на FashionMNIST.

**Дальше (видео 6):** `ImageFolder` — всё из блока 3 одной строкой, и датасет для регрессии (координаты из json).